# Challenge W4: NLP - Fake News
## Steps:
    1. Import data
    2. EDA
    3. Data Splitting
    4. Text Preprocessing 
    5. Feature Extraction
    6. Model Training

##  Libraries:

In [20]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
#import seaborn as sns

import re
import string

from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# 1. Load data

In [21]:
df = pd.read_csv("./dataset/data.csv")

In [22]:
df.shape
df.head()       
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39942 entries, 0 to 39941
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    39942 non-null  int64 
 1   title    39942 non-null  object
 2   text     39942 non-null  object
 3   subject  39942 non-null  object
 4   date     39942 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.5+ MB


# 2. EDA

In [23]:
df.isnull().sum() # missing values?

label      0
title      0
text       0
subject    0
date       0
dtype: int64

In [24]:
print("Duplicate rows:", df.duplicated().sum()) # Duplicates?


Duplicate rows: 201


In [25]:
df.sample(5, random_state=42)

,label,title,text,subject,date
6524,1,Oil business seen in strong position as Trump ...,(This January 3 story was corrected to remove...,politicsNews,"January 3, 2017"
30902,0,WHOA! COLLEGE SNOWFLAKE FREAKS OUT: Screams Fo...,So much for healthy debate on college campus I...,politics,"May 12, 2017"
36459,0,CRONY CORRUPT POLITICS: Obama Admin BLOCKED FB...,The information is spilling out little by litt...,Government News,"Aug 11, 2016"
9801,1,Cruz campaign vetting Fiorina as a possible VP...,WASHINGTON (Reuters) - U.S. Republican preside...,politicsNews,"April 25, 2016"
25638,0,Minnesota Woman Writes Amazing F*ck Off Lette...,"Attention, conservative men. This one is for y...",News,"July 2, 2016"


In [26]:
print("Unique subjects:", df['subject'].nunique())
df['subject'].value_counts()

Unique subjects: 6


subject
politicsNews       11272
News                9050
worldnews           8727
politics            6841
left-news           2482
Government News     1570
Name: count, dtype: int64

In [27]:
pd.crosstab(df['subject'], df['label']) # Because subject and label are correlated we should drop "subject" column to avoid data leakage.

label,0,1
subject,,
Government News,1570,0
News,9050,0
left-news,2482,0
politics,6841,0
politicsNews,0,11272
worldnews,0,8727


# 3. TRAIN TEST VALIDATION SPLIT

In [28]:
# Dropping columns subject and date as the have been shown to not carry important information
df = df.drop(columns=['subject','date']) 

In [29]:
df = df.drop_duplicates() # Dropping 201 duplicates that appeared once subject and date were removed (meaning that they had same title and/or text and we 
                            #could fall into data leakage)
print(df.shape) 

(36429, 3)


In [30]:

# Split into features and target
# Adjust the target column name to match your dataset (e.g. 'label', 'class', 'Spam/Ham')
X = df['title'] + ' ' + df['text']   # <-- replace 'label' with your actual target column
y = df['label']

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: Split the remaining 30% into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Display dataset shapes
print(f"Train shape:      {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape:       {X_test.shape}")

# Display class balance
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True))

print("\nValidation class balance:")
print(y_val.value_counts(normalize=True))

print("\nTest class balance:")
print(y_test.value_counts(normalize=True))

Train shape:      (25500,)
Validation shape: (5464,)
Test shape:       (5465,)

Train class balance:
label
1    0.543216
0    0.456784
Name: proportion, dtype: float64

Validation class balance:
label
1    0.543192
0    0.456808
Name: proportion, dtype: float64

Test class balance:
label
1    0.543275
0    0.456725
Name: proportion, dtype: float64


# 4. Text Preprocessing 
    - we will combine it into a reusable function 

In [31]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r'^[A-Z][A-Za-z\.,\'\s]*\(Reuters\)\s*-\s*', '', text)
    text = re.sub(r'\breuters\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\b(via|h/t)\s*:\s*\S.*$', '', text, flags=re.IGNORECASE)
    text = re.sub(r'read more\s*:\s*\S.*$', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bimage via\b.*$', '', text, flags=re.IGNORECASE)
    text = text.lower() # Lowercasing
    text = re.sub(r'[^a-z\s]', '', text) # Remove special characters
    tokens = word_tokenize(text) # Tokenization
    tokens = [word for word in tokens if word not in stop_words] # stopword removal
    tokens = [lemmatizer.lemmatize(word) for word in tokens] # Lemmatization
    return ' '.join(tokens)


Applying the text processing to X train, test and val

In [32]:
X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)
X_val_clean = X_val.apply(clean_text)

# 5. Feature extraction 
 - word embedding --> Word2Vec
 

In [33]:
train_tokens = X_train_clean.apply(lambda x: x.split())
val_tokens = X_val_clean.apply(lambda x: x.split())
test_tokens = X_test_clean.apply(lambda x: x.split())

In [36]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 9.5 MB/s  0:00:02m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]2m1/2 [gensim]


In [44]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=300,   # dimensionality of each word vector
    window=5,          # how many neighboring words considered as context
    min_count=2,        # ignore words appearing fewer than 2 times 
    workers=4,
    seed=42
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [45]:
def document_vector(tokens, model):
    valid_words = [word for word in tokens if word in model.wv]
    if len(valid_words) == 0:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[valid_words], axis=0)

X_train_w2v = np.array([document_vector(tokens, w2v_model) for tokens in train_tokens])
X_val_w2v = np.array([document_vector(tokens, w2v_model) for tokens in val_tokens])
X_test_w2v = np.array([document_vector(tokens, w2v_model) for tokens in test_tokens])

# 6. Model training - model_cf_2
    - Linear SVM 

In [46]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svm_w2v = LinearSVC(random_state=42, max_iter=5000)
svm_w2v.fit(X_train_w2v, y_train)

y_val_pred_w2v_svm = svm_w2v.predict(X_val_w2v)

print("=== Linear SVM + Word2Vec ===")
print("Accuracy:", accuracy_score(y_val, y_val_pred_w2v_svm))
print(classification_report(y_val, y_val_pred_w2v_svm))
print(confusion_matrix(y_val, y_val_pred_w2v_svm))


=== Linear SVM + Word2Vec ===
Accuracy: 0.9701683748169839
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      2496
           1       0.97      0.98      0.97      2968

    accuracy                           0.97      5464
   macro avg       0.97      0.97      0.97      5464
weighted avg       0.97      0.97      0.97      5464

[[2400   96]
 [  67 2901]]


In [48]:
coef_vector = svm_w2v.coef_[0]

vocab = w2v_model.wv.index_to_key
word_scores = [(word, np.dot(w2v_model.wv[word], coef_vector)) for word in vocab]
word_scores.sort(key=lambda x: x[1], reverse=True)

print("Words most aligned with predicting REAL (1):")
for word, score in word_scores[:20]:
    print(f"{word:20s} {score:.3f}")

print("\nWords most aligned with predicting FAKE (0):")
for word, score in word_scores[-20:][::-1]:
    print(f"{word:20s} {score:.3f}")

Words most aligned with predicting REAL (1):
said                 34.266
wednesday            33.490
monday               31.998
thursday             29.083
tuesday              28.974
friday               26.495
twitter              25.066
expressed            24.907
xi                   23.379
saturday             23.333
sunday               23.275
finance              23.214
concern              22.783
spokesman            22.537
hong                 22.156
kong                 22.069
province             21.922
opposition           21.893
spokeswoman          20.512
northern             19.999

Words most aligned with predicting FAKE (0):
lie                  -19.989
even                 -18.122
watch                -17.275
hillary              -16.737
fake                 -15.494
come                 -15.325
every                -14.431
look                 -13.870
truth                -13.565
go                   -13.492
terrorist            -13.255
pretty               -12.780
b

In [47]:
pd.set_option('display.max_colwidth', None)

errors = X_val[y_val != y_val_pred_w2v_svm]
print(errors.head(10))

33504                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   